In [1]:
# IN03: API vs MCP, Framework Selection, and Build vs Buy

In [2]:
# Load the libraries and keys needed for the full notebook.
# This cell also defines the store we will use in every example.
import os
import json
import time
import requests
from typing import TypedDict
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

OPENAI_KEY  = os.getenv('OPENAI_API_KEY')
WEATHER_KEY = os.getenv('OPENWEATHERMAP_API_KEY')
TAVILY_KEY  = os.getenv('TAVILY_API_KEY')

client = OpenAI(api_key=OPENAI_KEY)

# Check whether all required API keys are present before we continue.
missing = [k for k, v in {
    'OPENAI_API_KEY':         OPENAI_KEY,
    'OPENWEATHERMAP_API_KEY': WEATHER_KEY,
    'TAVILY_API_KEY':         TAVILY_KEY,
}.items() if not v]

if missing:
    print(f'WARNING: Missing API keys: {missing}')
    print('Add them to your .env file before running this notebook.')
else:
    print('All required API keys loaded.')

# Fixed store context keeps every comparison in the notebook consistent.
STORE_ID   = 'WMT-2847'
STORE_CITY = 'Bengaluru'
print(f'Store context: {STORE_ID} | {STORE_CITY}, India')

All required API keys loaded.
Store context: WMT-2847 | Bengaluru, India


In [3]:
# Walmart wants to build a production-ready AI Retail Assistant that can help store managers make decisions using live external information. The example used throughout the notebook is a Walmart store in Bengaluru, where the manager asks:
#     “Based on today’s actual conditions, what products should I prioritise stocking today?”

# The notebook therefore tries to answer three major engineering decisions:
# 1. REST API vs MCP:
# 2. Python-only vs LangChain vs LangGraph:
# 3. Build vs Buy

In [ ]:
# Build the same Walmart AI use case using different integration and orchestration approaches, compare them, and make an architecture decision about which approach should be used in production.

## The Decision Landscape

Every production AI system at Walmart scale requires three interlocking decisions made before any code is written:

1. **Protocol selection:** How should your AI agent communicate with external systems? REST API or Model Context Protocol (MCP)?
2. **Framework selection:** What orchestration layer do you build on? LangChain, LangGraph, or Python-only?
3. **Build vs Buy:** For each component, is it cheaper to build it or to buy a vendor solution?

Each decision compounds. A wrong protocol choice forces a framework rewrite downstream.

**The running scenario throughout this notebook:**

> *You are the AI engineer for the Walmart India Retail Assistant deployed at 4,700 stores with 50,000+ queries per day. The store manager at WMT-2847 in Bengaluru asks: "Based on today's actual conditions, what should we prioritise stocking today?"*

Every tool call in this notebook returns **live data** from real external APIs. The recommendation changes based on real weather and real market signals retrieved at runtime.

## Core API Functions

These two functions are the live data foundation used across all three sections.
Both make real HTTP calls to external services every time they are invoked.

In [ ]:
# def fetch_weather(city: str, country_code: str = 'IN') -> dict:
#     # Call the live OpenWeatherMap API for current weather.
#     url    = 'https://api.openweathermap.org/data/2.5/weather'
#     params = {'q': f'{city},{country_code}', 'appid': WEATHER_KEY, 'units': 'metric'}
#     resp   = requests.get(url, params=params, timeout=10)
#     resp.raise_for_status()
#     d = resp.json()
#     print(f"Weather API response: {json.dumps(d, indent=2)}")
    

# fetch_weather("Mumbai")

Weather API response: {
  "coord": {
    "lon": 72.8479,
    "lat": 19.0144
  },
  "weather": [
    {
      "id": 804,
      "main": "Clouds",
      "description": "overcast clouds",
      "icon": "04n"
    }
  ],
  "base": "stations",
  "main": {
    "temp": 28.99,
    "feels_like": 33.47,
    "temp_min": 28.99,
    "temp_max": 28.99,
    "pressure": 1011,
    "humidity": 74,
    "sea_level": 1011,
    "grnd_level": 1010
  },
  "visibility": 10000,
  "wind": {
    "speed": 3.09,
    "deg": 320
  },
  "clouds": {
    "all": 92
  },
  "dt": 1789063384,
  "sys": {
    "type": 1,
    "id": 9052,
    "country": "IN",
    "sunrise": 1789001725,
    "sunset": 1789046185
  },
  "timezone": 19800,
  "id": 1275339,
  "name": "Mumbai",
  "cod": 200
}


In [ ]:
# def fetch_demand_trends(query: str, max_results: int = 3) -> dict:
#     # Call the live Tavily API for current market demand signals.
#     resp = requests.post(
#         'https://api.tavily.com/search',
#         json={
#             'api_key':        TAVILY_KEY,
#             'query':          query,
#             'max_results':    max_results,
#             'search_depth':   'basic',
#             'include_answer': True,
#         },
#         timeout=15,
#     )
#     resp.raise_for_status()
#     data = resp.json()
#     print(f"Tavily API response: {json.dumps(data, indent=2)}")

# fetch_demand_trends("what are the latest updates with Nvidia?")

Tavily API response: {
  "query": "what are the latest updates with Nvidia?",
  "follow_up_questions": null,
  "answer": "Latest updates include new effects in NVIDIA Broadcast and 240 FPS ShadowPlay recording support. NVIDIA Update now offers automatic updates for game profiles. Recent acquisitions and partnerships include NVIDIA acquiring Hugging Face.",
  "images": [],
  "results": [
    {
      "url": "https://www.nvidia.com/en-us/software/nvidia-app/release-highlights",
      "title": "Updates and Release Highlights - NVIDIA",
      "content": "+ The latest NVIDIA Broadcast release features two new effects \u2013 the new Studio Voice, which upgrades your mic to deliver premium audio quality, and Virtual Keylight, which automatically relights your face for even lighting throughout your livestreams. Eye contact and background noise removal get a quality improvement, plus an updated user interface lets users combine even more effects. Download NVIDIA Broadcast from the Discover secti

In [6]:
# These helper functions are the live data layer used everywhere below.
# One gets weather data and the other gets demand signals from search.
def fetch_weather(city: str, country_code: str = 'IN') -> dict:
    # Call the live OpenWeatherMap API for current weather.
    url    = 'https://api.openweathermap.org/data/2.5/weather'
    params = {'q': f'{city},{country_code}', 'appid': WEATHER_KEY, 'units': 'metric'}
    resp   = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    d = resp.json()
    return {
        'city':           d['name'],
        'country':        d['sys']['country'],
        'temperature_c':  round(d['main']['temp'], 1),
        'feels_like_c':   round(d['main']['feels_like'], 1),
        'humidity_pct':   d['main']['humidity'],
        'condition':      d['weather'][0]['description'],
        'condition_main': d['weather'][0]['main'],
        'wind_speed_ms':  d['wind']['speed'],
        'pressure_hpa':   d['main']['pressure'],
    }


def fetch_demand_trends(query: str, max_results: int = 3) -> dict:
    # Call the live Tavily API for current market demand signals.
    resp = requests.post(
        'https://api.tavily.com/search',
        json={
            'api_key':        TAVILY_KEY,
            'query':          query,
            'max_results':    max_results,
            'search_depth':   'basic',
            'include_answer': True,
        },
        timeout=15,
    )
    resp.raise_for_status()
    data = resp.json()
    return {
        'answer':  data.get('answer', ''),
        'results': [
            {'title': r['title'], 'content': r['content'][:350]}
            for r in data.get('results', [])
        ],
    }


# Quick live check so we know both external services are working.
print('Verifying live API connections...')
print()

live_weather = fetch_weather(STORE_CITY)
print(f'OpenWeatherMap -- {live_weather["city"]}, {live_weather["country"]}:')
print(f'  Temperature : {live_weather["temperature_c"]} C  (feels like {live_weather["feels_like_c"]} C)')
print(f'  Condition   : {live_weather["condition"]}')
print(f'  Humidity    : {live_weather["humidity_pct"]}%   Wind: {live_weather["wind_speed_ms"]} m/s')

print()
live_demand = fetch_demand_trends(f'retail grocery product demand {STORE_CITY} India')
print(f'Tavily Search -- query returned {len(live_demand["results"])} results:')
if live_demand['answer']:
    print(f'  Synthesised: {live_demand["answer"][:220]}...')
for r in live_demand['results'][:2]:
    print(f'  - {r["title"][:80]}')

Verifying live API connections...

OpenWeatherMap -- Bengaluru, IN:
  Temperature : 26.6 C  (feels like 26.6 C)
  Condition   : overcast clouds
  Humidity    : 67%   Wind: 1.54 m/s

Tavily Search -- query returned 3 results:
  Synthesised: Bengaluru has strong demand for premium and specialty grocery products, driven by urban lifestyle trends. ZN Retail Pvt Ltd is a local retailer offering various grocery products. Ratnadeep Super Market is expanding its p...
  - Extract Urban vs Tier-2 Grocery Demand Data in India
  - ZN Retail Pvt Ltd in Bengaluru Karnataka - Retailer of Grocery Product & Soaps


## Section 1: REST API Integration -- Live External Services

In the REST pattern the developer manually writes a natural-language tool description and the AI uses that description to decide when and how to call the tool.

**Strengths:** Mature ecosystem, no new dependencies, works with any HTTP service.
**Weakness:** Tool schema lives in the developer's prompt. Schema drift and multi-agent reuse require copy-pasting descriptions across every agent that needs the tool.

The two tools below call **real external APIs** on every invocation:
- `get_store_weather` -- live call to OpenWeatherMap
- `search_demand_trends` -- live call to Tavily Search

In [7]:
# This cell shows the REST approach.
# The developer writes the tool schema by hand and the model uses it.
# REST tool schema -- developer writes this by hand for each agent that needs it
REST_TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'get_store_weather',
            'description': (
                'Get real-time weather at a Walmart India store location. '
                'Use this to identify weather-driven demand: '
                'rain drives umbrella/raincoat/waterproof footwear sales, '
                'heat drives cold beverages/ice cream/sunscreen sales, '
                'cold drives hot beverages/heaters/blanket sales.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'city':         {'type': 'string', 'description': 'City where the Walmart store is located'},
                    'country_code': {'type': 'string', 'description': 'ISO country code (default IN for India)'},
                },
                'required': ['city'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'search_demand_trends',
            'description': (
                'Search for real-time retail product demand trends and market signals using Tavily. '
                'Use this to identify high-demand product categories based on current market conditions.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'query':       {'type': 'string', 'description': 'Search query for demand or market signals'},
                    'max_results': {'type': 'integer', 'description': 'Number of results to return (1-5)'},
                },
                'required': ['query'],
            },
        },
    },
]

# LLM says:
# "Call get_store_weather"
#           ↓
# execute_rest_tool()
#           ↓
# Actual Python weather function
#           ↓
# External weather API
def execute_rest_tool(name: str, args: dict) -> str:
    # Route each REST tool call to the matching Python function.
    if name == 'get_store_weather':
        result = fetch_weather(args['city'], args.get('country_code', 'IN'))
    elif name == 'search_demand_trends':
        result = fetch_demand_trends(args['query'], args.get('max_results', 3))
    else:
        result = {'error': f'Unknown tool: {name}'}
    return json.dumps(result)


# Main Walmart REST agent
def walmart_rest_agent(query: str) -> dict:
    # Run a full REST-style agent loop with live tool calls.
    start    = time.time()
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are an AI assistant for Walmart India store {STORE_ID} in {STORE_CITY}. '
                'Use available tools to retrieve live data before making recommendations. '
                'Base your answer entirely on the real data returned by the tools.'
            ),
        },
        {'role': 'user', 'content': query},
    ]

    tools_called = []
    total_in = total_out = 0

# Because an agent interaction may look like:
# Iteration 1
# LLM → call weather

# Iteration 2
# LLM sees weather → call demand search

# Iteration 3
# LLM sees both results → produce final answer
# A single model call may not be sufficient.
# The maximum of six prevents an infinite loop.

    # Keep calling the model until it stops asking for tools.
    for _ in range(6):
        resp = client.chat.completions.create(
            model='gpt-4o-mini', messages=messages, tools=REST_TOOLS,
            tool_choice='auto', temperature=0, max_tokens=500,
        )
        total_in  += resp.usage.prompt_tokens
        total_out += resp.usage.completion_tokens
        msg = resp.choices[0].message

        if not msg.tool_calls:
            final_answer = msg.content.strip()
            break

        messages.append(msg)
        for tc in msg.tool_calls:
            args   = json.loads(tc.function.arguments)
            result = execute_rest_tool(tc.function.name, args)
            tools_called.append({'tool': tc.function.name, 'args': args})
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})

    latency = time.time() - start
    cost    = (total_in * 0.15 + total_out * 0.60) / 1_000_000
    return {
        'answer':       final_answer,
        'tools_called': tools_called,
        'latency_sec':  round(latency, 2),
        'cost_usd':     round(cost, 6),
        'tokens_in':    total_in,
        'tokens_out':   total_out,
        'protocol':     'REST',
    }

In [8]:
# Ask the REST agent the main Walmart store question and inspect the result.
STORE_QUERY = (
    f'I am the store manager at {STORE_ID} in {STORE_CITY}. '
    "Based on today's actual weather conditions and current market demand signals, "
    'give me 3 specific product categories I should prioritise stocking today. '
    'For each category, explain why the live data supports this recommendation.'
)

print(f'Query: {STORE_QUERY}')
print()

rest_result = walmart_rest_agent(STORE_QUERY)

# Show which tools were used before we read the final answer.
print('Tools called by the REST agent:')
for t in rest_result['tools_called']:
    print(f'  {t["tool"]}({json.dumps(t["args"])})')

print()
print('REST Agent Answer (based on live data):')
print('-' * 70)
print(rest_result['answer'])
print('-' * 70)
print(f'Latency : {rest_result["latency_sec"]}s')
print(f'Cost    : ${rest_result["cost_usd"]}')
print(f'Protocol: REST (hand-written tool schema)')

Query: I am the store manager at WMT-2847 in Bengaluru. Based on today's actual weather conditions and current market demand signals, give me 3 specific product categories I should prioritise stocking today. For each category, explain why the live data supports this recommendation.

Tools called by the REST agent:
  get_store_weather({"city": "Bengaluru", "country_code": "IN"})
  search_demand_trends({"query": "current market demand", "max_results": 5})

REST Agent Answer (based on live data):
----------------------------------------------------------------------
Based on the current weather conditions and market demand signals in Bengaluru, here are three specific product categories to prioritize stocking today:

1. **Cold Beverages**:
   - **Weather Support**: The current temperature is around 26.7°C with a "feels like" temperature of 28.1°C. This warm weather typically drives demand for cold beverages such as soft drinks, juices, and bottled water.
   - **Market Demand**: While spec

## Section 2: Model Context Protocol (MCP) -- Real FastMCP Server

MCP (Anthropic, 2024) separates tool definition from tool calling.
The MCP server owns the schema. Any MCP-compatible AI client connects, discovers tools automatically, and calls them -- without the developer writing a single word of tool description.

**Architecture:**
```
AI Host (GPT-4o-mini)  <-->  MCP Client (test_client)  <-->  FastMCP Server  <-->  Live External APIs
```

**What changes vs REST:**
- Tool descriptions live on the **server**, not in developer prompts
- Schema is **machine-readable** JSON -- no natural language required
- Any number of AI clients can point at the same server without duplicating schema
- The server handles versioning; clients auto-discover changes on reconnect

**What stays the same:**
- Underlying HTTP calls to OpenWeatherMap and Tavily are identical
- OpenAI token cost per call is the same
- MCP adds a schema-fetch round-trip on first connection (~5ms)

The FastMCP server below uses real API calls inside every registered tool.

In [9]:
# What problem do we already have with REST?

# Walmart AI Agent
#    ↓
# Developer manually describes:
#    - Weather tool
#    - Tool parameters
#    - What the tool does
#    ↓
# LLM decides to call weather
#    ↓
# Python code routes the call
#    ↓
# OpenWeatherMap REST API

# Perfectly valid.
# Suppose Walmart has only:
# 1 Agent
# 2 Tools
# No major problem.

# But now imagine Walmart grows to:
# 20 AI agents
# 50 enterprise tools
# Different development teams
# Different APIs
# Different authentication mechanisms
# Different schemas

# Now every agent may have to repeatedly define things such as:
# What is this tool?
# What parameters does it accept?
# How do I call it?
# What data does it return?
# That is the problem MCP is trying to standardize.

In [10]:
# MCP = Model Context Protocol.
# It is an open standard for connecting AI applications to external tools, data and systems in a common way.

# Years ago, different devices could require different connectors.
# Then USB gave us a standard.

# Keyboard ─┐
# Mouse ────┤
# Camera ───┤
# Drive ────┤
#           ↓
#        USB standard
#           ↓
#        Computer

# The computer doesn't need a completely different connection philosophy for every device.
# MCP is trying to provide something similar for AI:

# Weather system ─────┐
# Database ───────────┤
# GitHub ─────────────┤
# CRM ────────────────┤
# Search service ─────┤
#                     ↓
#                    MCP
#                     ↓
#                AI application

# MCP was originally created at Anthropic.
# Anthropic announced and open-sourced MCP on November 25, 2024. Anthropic specifically credits David Soria Parra and Justin Spahr-Summers with creating MCP.

In [11]:
# Why was MCP invented?
# AI models were becoming very powerful at reasoning, but they were still isolated from real-world systems.

# Every time developers wanted to connect an AI application to:
# Google Drive
# Slack
# GitHub
# Database
# CRM
# Search engine
# Internal API
# they often had to build a separate custom integration.

# Anthropic described this as a scalability problem caused by fragmented integrations and information silos. MCP was introduced as a common protocol for connecting AI applications with these systems.

In [12]:
# REST vs MCP?

# Do not think:
# REST OR MCP
# as though one completely replaces the other.

# Very often it can actually be:
# LLM
#  ↓
# MCP
#  ↓
# Your MCP Server
#  ↓
# REST API
#  ↓
# External service

# ex:
# Walmart Agent
#       ↓
# MCP
#       ↓
# Weather MCP Server
#       ↓
# OpenWeatherMap REST API

# So REST may still exist underneath.
# MCP is providing a standardized AI-facing integration layer.
# That distinction is extremely important.

In [15]:
# The architecture is:

# HOST
#   ↓
# MCP CLIENT
#   ↓
# MCP SERVER
#   ↓
# External system/tool/data

# What is the MCP Host?
# The Host is the AI application the user interacts with.
# ex:
# AI desktop application
# IDE
# Enterprise AI application
# Your Walmart AI assistant

# Store Manager
#      ↓
# Walmart Retail Assistant
#         HOST
#      ↓
# GPT / Claude / other LLM
# The host is responsible for broader concerns such as permissions, lifecycle and coordinating the AI interaction.

# What is an MCP Client?
# An MCP Client is a component inside the host that communicates with an MCP Server using the MCP protocol.

# Host = Walmart AI application

# Inside Host:

# MCP Client
#      ↓
# communicates using MCP
#      ↓
# MCP Server


# What is an MCP Server?
# An MCP Server exposes capabilities to AI applications in MCP format.
# Those capabilities commonly include:
# Tools
# Resources
# Prompts

In [16]:
# Full Walmart architecture:

# STORE MANAGER
#      │
#      │ "What should I stock today?"
#      ↓
# WALMART RETAIL ASSISTANT
#         MCP HOST
#      │
#      ↓
#        LLM
#      │
#      ↓
#     MCP CLIENT
#      │
#      │ MCP communication
#      ↓
#    MCP SERVER
#  ┌──────────────────────┐
#  │ get_store_weather    │
#  │ search_demand_trends │
#  │ check_inventory      │
#  └──────────┬───────────┘
#             ↓
#        Real systems
#    ┌────────┼─────────┐
#  Weather   Tavily   Inventory DB

In [17]:
# This cell sets up the FastMCP server and registers its tools.
# The main idea is that tool definitions live on the server side.
MCP_AVAILABLE = False

try:
    import inspect # inspect is a standard Python module that allows Python code to examine other Python functions.
    # Why is that useful here?
    # Consider:
    # def get_store_weather(city: str, country_code: str = 'IN') -> dict:
    # inspect can discover:
    # Function name        → get_store_weather
    # Parameters           → city, country_code
    # Types                → str, str
    # Default              → IN
    # Required parameters  → city
    # Documentation        → docstring

    from mcp.server.fastmcp import FastMCP

    # This helps async-friendly packages behave inside Jupyter.
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        pass

    walmart_mcp = FastMCP(
        'walmart-store-ops',
        instructions=(
            'Walmart India Store Operations MCP Server. '
            'Exposes real-time weather intelligence and market demand signals '
            'for store management decisions at WMT stores across India.'
        ),
    )

    @walmart_mcp.tool()
    def get_store_weather(city: str, country_code: str = 'IN') -> dict:
        """Get current weather at a Walmart India store location for demand planning."""
        return fetch_weather(city, country_code)

    @walmart_mcp.tool()
    def search_demand_trends(query: str, max_results: int = 3) -> dict:
        """Search live retail demand trends and market signals for store planning."""
        return fetch_demand_trends(query, max_results)

    @walmart_mcp.tool()
    def get_store_info(store_id: str) -> dict:
        """Retrieve operational metadata for a Walmart India store."""
        registry = {
            'WMT-2847': {'location': 'Bengaluru, Karnataka', 'format': 'Supercenter',
                         'departments': 32, 'daily_queries': 1200, 'region': 'South India'},
            'WMT-1023': {'location': 'Mumbai, Maharashtra',  'format': 'Supercenter',
                         'departments': 28, 'daily_queries': 1800, 'region': 'West India'},
            'WMT-0511': {'location': 'Delhi NCR',            'format': 'Supercenter',
                         'departments': 35, 'daily_queries': 2100, 'region': 'North India'},
            'WMT-3302': {'location': 'Hyderabad, Telangana', 'format': 'Neighborhood Market',
                         'departments': 18, 'daily_queries':  620, 'region': 'South India'},
        }
        return registry.get(store_id, {'error': f'Store {store_id} not found in registry'})

    def _annotation_to_json_type(annotation) -> str:
        # Convert Python type hints into simple JSON schema types.
        return {
            str: 'string',
            int: 'integer',
            float: 'number',
            bool: 'boolean',
            dict: 'object',
            list: 'array',
        }.get(annotation, 'string')

    def _build_mcp_tool_entry(func) -> dict:
        # Build a small machine-readable schema for each MCP tool.
        sig = inspect.signature(func)
        properties = {}
        required = []

        for param_name, param in sig.parameters.items():
            if param.kind not in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY):
                continue

            prop = {'type': _annotation_to_json_type(param.annotation)}
            if param.default is not inspect._empty:
                prop['default'] = param.default
            else:
                required.append(param_name)
            properties[param_name] = prop

        return {
            'name': func.__name__,
            'description': inspect.getdoc(func) or '',
            'inputSchema': {
                'type': 'object',
                'properties': properties,
                'required': required,
            },
            'handler': func,
        }

    # Keep the registered MCP tools in one place for easy lookup below.
    MCP_TOOL_REGISTRY = {
        tool['name']: tool
        for tool in [
            _build_mcp_tool_entry(get_store_weather),
            _build_mcp_tool_entry(search_demand_trends),
            _build_mcp_tool_entry(get_store_info),
        ]
    }

    def call_mcp_tool(name: str, args: dict) -> dict:
        # Run an MCP tool by name using the saved registry entry.
        return MCP_TOOL_REGISTRY[name]['handler'](**args)

    MCP_AVAILABLE = True
    print(f'FastMCP server created: {walmart_mcp.name}')
    print()
    print('Tools registered on the MCP server:')
    print('  - get_store_weather    (live data: OpenWeatherMap API)')
    print('  - search_demand_trends (live data: Tavily Search API)')
    print('  - get_store_info       (store registry -- note: 3 tools vs 2 for REST)')
    print()
    print('Key difference from REST: tool descriptions live here on the server,')
    print('not in the developer\'s agent code. Any MCP client discovers them automatically.')

except ImportError as e:
    print(f'mcp package not installed: {e}')
    print('Install: pip install mcp --break-system-packages')
    print()
    print('The REST section above is fully functional without this package.')
    print('FastMCP cells below will be skipped gracefully.')

FastMCP server created: walmart-store-ops

Tools registered on the MCP server:
  - get_store_weather    (live data: OpenWeatherMap API)
  - search_demand_trends (live data: Tavily Search API)
  - get_store_info       (store registry -- note: 3 tools vs 2 for REST)

Key difference from REST: tool descriptions live here on the server,
not in the developer's agent code. Any MCP client discovers them automatically.


In [19]:
# Now combine MCP tool schemas, MCP tool execution, and the LLM in one loop.
# This mirrors the REST agent flow, but the tool schema comes from the server side.
# Step 3: Full MCP Agent -- schema from server registry, tool execution from MCP server definitions

if MCP_AVAILABLE:
    def walmart_mcp_agent(query: str) -> dict:
        # Run a full MCP-style agent loop using the server-owned schema.
        start = time.time()

        openai_tools = [
            {
                'type': 'function',
                'function': {
                    'name':        t['name'],
                    'description': t['description'],
                    'parameters':  t['inputSchema'],
                },
            }
            for t in MCP_TOOL_REGISTRY.values()
        ]

        messages = [
            {
                'role': 'system',
                'content': (
                    f'You are an AI assistant for Walmart India store {STORE_ID} in {STORE_CITY}. '
                    'Tools are provided by the Walmart MCP server. '
                    'Use all relevant tools to give a complete, data-driven recommendation.'
                ),
            },
            {'role': 'user', 'content': query},
        ]

        tools_called = []
        total_in = total_out = 0
        final_answer = ''

        # Keep going until the model stops asking for tool calls.
        for _ in range(6):
            resp = client.chat.completions.create(
                model='gpt-4o-mini', messages=messages, tools=openai_tools,
                tool_choice='auto', temperature=0, max_tokens=500,
            )
            total_in  += resp.usage.prompt_tokens
            total_out += resp.usage.completion_tokens
            msg = resp.choices[0].message

            if not msg.tool_calls:
                final_answer = msg.content.strip()
                break

            messages.append(msg)

            for tc in msg.tool_calls:
                args   = json.loads(tc.function.arguments)
                result = call_mcp_tool(tc.function.name, args)
                tools_called.append({'tool': tc.function.name, 'args': args})
                messages.append({
                    'role':         'tool',
                    'tool_call_id': tc.id,
                    'content':      json.dumps(result),
                })

        latency = time.time() - start
        cost    = (total_in * 0.15 + total_out * 0.60) / 1_000_000
        return {
            'answer':       final_answer,
            'tools_called': tools_called,
            'latency_sec':  round(latency, 2),
            'cost_usd':     round(cost, 6),
            'tokens_in':    total_in,
            'tokens_out':   total_out,
            'protocol':     'MCP',
        }

    mcp_result = walmart_mcp_agent(STORE_QUERY)

    print('Tools called via MCP protocol:')
    for t in mcp_result['tools_called']:
        print(f'  {t["tool"]}({json.dumps(t["args"])})')
    print()
    print('MCP Agent Answer (based on live data):')
    print('-' * 70)
    print(mcp_result['answer'])
    print('-' * 70)
    print(f'Latency : {mcp_result["latency_sec"]}s')
    print(f'Cost    : ${mcp_result["cost_usd"]}')
    print(f'Protocol: MCP (schema auto-discovered from FastMCP server)')

else:
    mcp_result = {**rest_result, 'protocol': 'MCP (fallback -- install mcp package)'}
    print('mcp package not installed. Showing REST result as reference.')
    print(f'Answer: {mcp_result["answer"][:300]}...')

[09/11/26 00:31:43] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=750241;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=6162;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[09/11/26 00:31:47] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=576122;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=107188;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[09/11/26 00:31:54] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=630944;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=343620;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Tools called via MCP protocol:
  get_store_weather({"city": "Bengaluru"})
  search_demand_trends({"query": "Bengaluru", "max_results": 3})
  search_demand_trends({"query": "weather", "max_results": 3})

MCP Agent Answer (based on live data):
----------------------------------------------------------------------
Based on the current weather conditions in Bengaluru and the latest market demand signals, here are three specific product categories you should prioritize stocking today:

### 1. **Rain Gear (Umbrellas, Raincoats)**
- **Reasoning**: The weather in Bengaluru is currently overcast with a temperature of 26.7°C and a humidity level of 67%. This suggests a likelihood of rain, which typically increases the demand for rain gear. Customers are likely to seek out umbrellas and raincoats to stay dry, especially if there are scattered showers expected throughout the day.

### 2. **Comfort Foods (Snacks, Instant Noodles, Soups)**
- **Reasoning**: Overcast and humid weather often leads to a

In [18]:
STORE_QUERY

"I am the store manager at WMT-2847 in Bengaluru. Based on today's actual weather conditions and current market demand signals, give me 3 specific product categories I should prioritise stocking today. For each category, explain why the live data supports this recommendation."

In [ ]:
# To be continued...